# Notebook 05 — Build a Semantic Search Engine

> **Easiest way to run this: Google Colab — nothing to install.**
> Go to https://colab.research.google.com → **File > Upload notebook** → choose this file.
> Prefer your own computer? Lesson 1 shows the VS Code and local-Jupyter paths too.

This is the payoff. We build a small **semantic search engine**: a pile of documents, and a
`search()` function that finds the ones closest in *meaning* to whatever you ask — even when
your words and the document's words don't overlap at all.

Every piece here you've already met: embeddings (Notebook 03) and a vector store (Notebook
04). We're just assembling them.

In [ ]:
%pip install -q sentence-transformers chromadb
print("Ready.")

## Step 1 — A small library of documents

Twenty short documents on a mix of topics: space, cooking, money, health, and tech.

In [ ]:
documents = [
    # Space
    "Astronauts aboard the space station experience weightlessness for months.",
    "A telescope gathers faint light from distant galaxies.",
    "The rover collected rock samples from the surface of Mars.",
    "Comets grow a glowing tail as they near the sun.",
    # Cooking
    "Let the bread dough rise for an hour before baking.",
    "Simmer the tomatoes slowly to deepen the sauce's flavour.",
    "Whisk the eggs until the mixture turns pale and fluffy.",
    "Season the soup with a pinch of salt and fresh herbs.",
    # Money
    "Paying off high-interest debt first saves you the most money.",
    "A diversified portfolio spreads risk across many assets.",
    "Compound interest grows your savings faster over time.",
    "Set aside an emergency fund before you start investing.",
    # Health
    "Regular walking lowers blood pressure and lifts your mood.",
    "Drinking enough water keeps you alert through the afternoon.",
    "Stretching before exercise reduces the chance of injury.",
    "A good night's sleep helps your body repair itself.",
    # Technology
    "The new laptop boots in seconds thanks to its fast drive.",
    "Encryption keeps your messages private from eavesdroppers.",
    "Cloud storage lets you reach your files from any device.",
    "A strong password is long, unusual, and hard to guess.",
]
ids = [f"doc_{i}" for i in range(len(documents))]
print(f"{len(documents)} documents ready.")

## Step 2 — Put them in a vector store

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

minilm_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
client = chromadb.PersistentClient(path="./search_store")
try:
    client.delete_collection("library")
except Exception:
    pass
library = client.create_collection(
    name="library",
    embedding_function=minilm_ef,
    metadata={"hnsw:space": "cosine"},
)
library.add(ids=ids, documents=documents)
print(f"Indexed {library.count()} documents.")

## Step 3 — The search function

Three lines of real work: hand the query to the store, get back the closest documents, print
them with a similarity score.

In [ ]:
def search(query, k=3):
    result = library.query(query_texts=[query], n_results=k)
    print(f"{query!r}\n")
    for doc, dist in zip(result["documents"][0], result["distances"][0]):
        print(f"  similarity {1 - dist:.3f}   {doc}")
    print()

search("how do I keep my data safe online?")

## Step 4 — The magic: no shared words

Watch each query find the right documents even though it shares almost no words with them.

In [ ]:
search("what helps me relax and feel rested?")   # -> sleep / walking / water
search("growing my money for the future")         # -> investing / compound interest
search("exploring other planets")                 # -> Mars rover / telescope / comets

## Step 5 — Your turn, and recap

**Try it:** call `search("...")` with your own questions. Try one that shares zero words with
any document and watch it still find the right topic.

**Recap**
- A semantic search engine = embeddings + a vector store + a tiny `search()` wrapper.
- It matches meaning, so it finds documents that share *ideas*, not just words.
- You built the whole thing from parts you already understood.

**Next (Notebook 06):** a mini challenge — you build a Smart FAQ Finder mostly on your own,
with hints if you need them.